# v4 파이프라인 단계별 검수 — 질문마다 근거 카드 → 어느 단계가 문제인지 판정

`kosis_rag_eval_80_queries.csv` 의 **질문 문장만** 넣어 v4 파이프라인을 돌리고, 질문마다 **카드 한 장(HTML + PNG)** 에
단계별 근거를 모은다. 카드를 보고 단계마다 PASS/PARTIAL/FAIL 과 **첫 실패 단계**·코멘트를 남기면 문제 단계가 집계된다.

| 카드 구역 | 근거 |
|---|---|
| ① 쿼리 분석 | 재작성(v3) 문장 · HCX 슬롯 · **슬롯 → CSV 값 접지** (출처 rule/csv/term>csv · 그 값을 가진 표·CSV 값) · 시점 규칙 · 되묻기(자동 응답) · 서브쿼리 · 검색 문장 |
| ② 검색 | 검색 단위마다 후보 10개: 순위 · 점수 · BM25 순위 · **못 맞춘 조건**(빨간 행) · View 생성 여부 |
| ③ View | View 마다 **KOSIS 통계표 링크 · 원본 CSV · View CSV** · 필수 메타 사용(판단 근거) · 못 맞춘 조건 · **잘라 낸 표 ↔ 원본 CSV 같은 좌표 행 값 대조** |
| ④ 충돌·선택 | 같은 교차점 값이 다른 표 · 선택지(주석·산출 기준·KOSIS) · 자동 선택 |
| ⑤ 답변 | (선택) HCX 답변 · 근거에 없는 숫자 |

자동 진단은 **초안**이다 (사유 코드: `q.clarify`, `s.none_ok`, `v.raw_mismatch`, `c.ambiguous` …). 사람 판정이 최종.

| 셀 | 하는 일 | 결과 (`storage/eval_runs/kosis_rag_eval_80_queries__v4/`) |
|---|---|---|
| 1 | 설정 · 테스트 세트 확인 | |
| 2 | 자동 실행 (이어서 실행) | `results.json` · `json/Q001.json` · `cards/Q001.html·png` · `review.csv` |
| 3 | 신호등 표 (단계별 자동·사람) | |
| 4 | 단계별 문제 집계 · 사유별 문항 | |
| 5·6 | 카드 보기 → 단계별 판정 저장 | 위 파일 모두 갱신 |

코드: `eval/pipeline_review/review_v4.py` (CLI 도 있음: `python eval/pipeline_review/review_v4.py run --queries 80`)

In [ ]:
# [Cell 1] 설정
import importlib, os, sys, webbrowser
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend").is_dir())     # 저장소 루트
for p in (ROOT, ROOT / "eval" / "pipeline_review"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
os.chdir(ROOT)
USE_FAKE = os.getenv("V4_NB_FAKE", "0") == "1"      # 데이터·HCX 없이 구조만 (가짜 데이터 + HCX 대역)
if USE_FAKE:
    import _fake_setup; _fake_setup.setup(ROOT)

import pandas as pd
from IPython.display import HTML, Image, display
import review_v4 as rv
importlib.reload(rv)
pd.set_option("display.max_colwidth", 80)

rv.list_sets()                                   # TEST_DATA_DIR 에서 찾은 테스트 세트 (번호·문항 수·결과 폴더·완료 수)

QUERIES = "80"            # 테스트 세트: 번호 | 파일 이름 일부 | 경로 (예: "test_data/kosis_rag_eval_80_queries.csv")
SETTINGS = dict(
    today="2026-11-02",   # '작년'·'최근' 기준일 (평가셋 기준일로 고정)
    clarify="first",      # 되묻기: first = 첫 후보로 자동 응답 (카드에 표시) | none = 되묻기에서 멈춤
    select="first",       # 충돌 선택: first = 검색 순위 높은 선택지 자동 (카드에 표시) | none = 선택 대기
    n_views=3,            # 단위마다 View 를 만들어 비교할 표 수
    retrieval_config=None,  # 검색 구성 (None = 기존 검색 = baseline). 전체 임베딩 구성은 처음에 오래 걸림
    answer=False,         # ⑤ HCX 답변까지 (문항당 HCX 1회 추가)
    raw=True,             # 원본 CSV 같은 좌표 행 대조
    overrides={"csv_scope": "all"} if USE_FAKE else None,   # configs/query_v4.yaml 일부 바꾸기
)
RUN, TS = rv.run_for(QUERIES)
print("질문 파일:", TS["path"], "\n결과 폴더:", RUN.dir)

---
## [Cell 2] 자동 실행
- HCX: 문항당 쿼리 분석 1회 (+ `answer=True` 면 1회) + 후보 표 임베딩(캐시)
- 매 문항 저장 → 멈춰도 다시 실행하면 **안 끝난 문항만** 이어서. `FORCE=True` 면 다시 (사람 판정은 유지)
- 처음엔 `N = 5` 로 확인 → `N = None` 으로 전체

In [ ]:
N = 5            # None = 전체
IDS = None       # 예: ["Q003", "Q017"]
FORCE = False
PNG = True       # 카드 PNG 도 (playwright 또는 html2image 필요, 없으면 HTML 만)

RUN, DATA = rv.run_batch(QUERIES, ids=IDS, n=N, force=FORCE, png=PNG, **SETTINGS)

---
## [Cell 3] 신호등 표 — 단계별 자동(초안) · 사람 판정
`VIEW`: `"ALL"` · `"PROBLEM"`(자동 PASS 아닌 단계가 있는 문항) · `"UNREVIEWED"` · 단계 이름(`"search"` 등 = 그 단계가 PASS 아닌 문항)

In [ ]:
VIEW = "ALL"
DATA = rv.load_results(RUN)
COL = {"PASS": "#d4edda", "PARTIAL": "#fff3cd", "FAIL": "#f8d7da", "CHECK": "#dbeafe"}
def _keep(r):
    a = r["auto"]
    if VIEW == "PROBLEM":
        return any((a.get(s) or {}).get("grade") not in ("PASS", "-", None) for s in rv.STAGES)
    if VIEW == "UNREVIEWED":
        return not r["human"].get("reviewed")
    if VIEW in rv.STAGES:
        return (a.get(VIEW) or {}).get("grade") not in ("PASS", "-", None)
    return True
df = rv.review_df({k: v for k, v in DATA.items() if _keep(v)})
cols = ["id", "question", "분석_상태"] + [c for c in df.columns if c.startswith(("자동_①", "자동_②", "자동_③", "자동_④", "사람_"))] \
       + ["자동_첫_문제_단계", "사람_첫_실패_단계", "최종_상태", "검수", "초"]
cols = [c for c in cols if c in df.columns and not (df[c] == "").all() or c in ("id", "question")]
print(f"전체 {len(DATA)}문항 · 표시 {len(df)} ({VIEW}) · 검수 완료 {sum(r['human'].get('reviewed', False) for r in DATA.values())}")
sty = df[cols].style
display(getattr(sty, "map", None) and sty.map(lambda v: f"background:{COL.get(v, '')}" if v in COL else "")
        or sty.applymap(lambda v: f"background:{COL.get(v, '')}" if v in COL else ""))   # pandas 2.1 미만은 applymap

---
## [Cell 4] 단계별 문제 집계 — 어느 파이프라인이 얼마나, 왜
- 위 표: 단계별 판정 분포 (`WHO="auto"` 자동 초안 · `"human"` 사람 판정)
- 아래 표: 사유 코드별 문항 수 → `rv.questions_with(DATA, "코드")` 로 해당 문항 id

In [ ]:
WHO = "auto"
gt, rt = rv.stage_summary(DATA, WHO)
display(gt); display(rt)
CODE = "s.none_ok"        # 예: q.clarify · q.ungrounded · s.none_ok · s.top1_unmet · v.unusable · v.raw_mismatch · c.ambiguous
print(CODE, "→", rv.questions_with(DATA, CODE))

---
## [Cell 5] 카드 보기
`QID` 를 바꿔 가며. 카드의 KOSIS·원본 CSV·View CSV 는 링크 (HTML 카드에서 클릭).
노트북 안에서 링크가 안 열리면 `open_kosis(QID, 1)` · `open_raw(QID, 1)` · `open_card(QID)`.

In [ ]:
QID = "Q001"

def _views(qid):
    return [v for u in DATA[qid].get("units") or [] for v in u["views"]]

def open_kosis(qid, n=1):
    webbrowser.open(_views(qid)[n - 1]["kosis_url"])

def open_raw(qid, n=1):
    p = (_views(qid)[n - 1].get("raw") or {}).get("path")
    os.startfile(p) if hasattr(os, "startfile") else webbrowser.open(Path(p).as_uri())

def open_card(qid):
    webbrowser.open((RUN.card_dir / f"{qid}.html").as_uri())

DATA = rv.load_results(RUN)
display(HTML(rv.card_html(DATA[QID])))
for i, v in enumerate(_views(QID), 1):
    print(f"  View {i}: {v['tbl_id']}  KOSIS {v['kosis_url']}  원본 {(v.get('raw') or {}).get('path')}")

---
## [Cell 6] 👇 단계별 판정 저장
`None` 이면 그대로. 저장하면 카드(HTML·PNG)·`results.json`·`review.csv` 가 갱신된다.
첫 실패 단계: `"query"` · `"search"` · `"view"` · `"conflict"` · `"answer"` · `""`(문제 없음)

In [ ]:
QID = "Q001"
GRADE = dict(query=None, search=None, view=None, conflict=None, answer=None)   # "PASS" | "PARTIAL" | "FAIL" | "CHECK"
FIRST_FAIL = None        # 예: "search"
COMMENT = None           # 예: "1위가 외국인 포함 표 — 주민등록 기준 표가 3위"

rec = rv.set_review(RUN, QID, first_fail=FIRST_FAIL, comment=COMMENT, png=PNG, **GRADE)
DATA = rv.load_results(RUN)
display(HTML(rv.card_html(DATA[QID])))
print("💾 저장 →", RUN.dir / "review.csv")

---
## [Cell 7] 내보내기
- `review.csv` (엑셀에서 열림): 문항별 단계 자동·사람 판정, 첫 실패 단계, 코멘트, 자동 사유, 1위 View 표·KOSIS 링크
- `cards/` 폴더: 카드 HTML·PNG (공유·보고서용). PNG 를 다시 만들려면 아래

In [ ]:
# rv.write_cards(RUN, png=True)          # 카드 전체 다시 만들기
print(RUN.dir / "review.csv"); print(RUN.card_dir)